# CivLM · Colab 训练（A100）

流程：上传代码 → 生成数据 → 训练 → 打包下载 → 本地评测。

先在菜单 **代码执行程序 → 更改运行时类型** 里选择 **A100 GPU**，然后从上往下依次运行。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, multiprocessing
print("CPU 核数:", multiprocessing.cpu_count())

## 1. 写入代码（代码已内嵌在笔记本里，无需上传）


In [ ]:
import os
os.makedirs('/content/civlm', exist_ok=True)
%cd /content/civlm
!pip -q install tokenizers


In [ ]:
%%writefile /content/civlm/engine.py
"""
CivEngine v0：确定性的文明模拟器
================================
- 1 个文明，每年：玩家/策略选择一个行动 → 外部天气 → 结算收成、贸易、人口、情绪。
- 每个世界有 3 个【隐藏规则】，语言模型永远看不到，只能从历史中推断：
    irrigation  灌溉   boost    每条水渠 +30% 产量
                       salinize 同样 +30%，但土壤逐年盐碱化，产量随后下降
                       weak     每条水渠只 +5%
    climate     气候   wet      干旱概率 5%
                       dry      干旱概率 35%
    trade       贸易   rich     30 木材换约 66 粮食
                       poor     30 木材换约 21 粮食
  另有连续的隐藏参数（单块农田基础产量、贸易汇率的 ±15% 抖动），防止模型只背下 12 种世界。
- 完全可复现：同一 (seed, rules) 永远生成同一段历史。
- 天气序列事先由种子抽好（外部事件 E_t），因此可以从任意一年复制世界做反事实分叉，
  两个分支面对完全相同的天气。
"""
import copy
import itertools
import math
import random
from dataclasses import asdict, dataclass

RULES = {
    "irrigation": ["boost", "salinize", "weak"],
    "climate": ["wet", "dry"],
    "trade": ["rich", "poor"],
}
ALL_COMBOS = [dict(zip(RULES, v)) for v in itertools.product(*RULES.values())]
# 训练中完全不出现的规则组合（每个规则取值单独都见过，只是组合没见过 = 组合泛化）
HELDOUT_COMBOS = [
    {"irrigation": "salinize", "climate": "dry", "trade": "rich"},
    {"irrigation": "boost", "climate": "wet", "trade": "poor"},
]
TRAIN_COMBOS = [c for c in ALL_COMBOS if c not in HELDOUT_COMBOS]

IRR_GAIN = {"boost": 0.30, "salinize": 0.30, "weak": 0.05}
P_DROUGHT = {"wet": 0.05, "dry": 0.35}
P_BOUNTIFUL = 0.12
TRADE_RATE = {"rich": 2.2, "poor": 0.7}
BASE_YIELD = 45.0
SALT_PER_CANAL = 0.06      # 盐碱化：每条水渠每年使土壤肥力下降 0.06
SOIL_MIN = 0.35
SOIL_RECOVER = 0.05
YIELD_NOISE = 0.10         # 每年产量的随机波动（外部因素，分叉时保持一致）

ACTIONS = ["farm", "canal", "forest", "trade", "research", "feast", "house", "rest"]
WEATHERS = ["normal", "drought", "bountiful"]
COST = {"farm": 25, "canal": 35, "house": 30, "trade": 30}
MAX_CANALS, MAX_HOUSES, MAX_TECH = 3, 10, 4
PEOPLE_PER_FARM = 25
HOUSE_CAP = 50


@dataclass
class Params:               # 隐藏的世界参数
    irrigation: str
    climate: str
    trade: str
    y0: float               # 单块农田基础产量
    trade_rate: float       # 每 1 木材换多少粮食

    @property
    def rules(self):
        return {"irrigation": self.irrigation, "climate": self.climate, "trade": self.trade}


@dataclass
class State:
    year: int
    pop: int
    food: int
    wood: int
    farms: int
    canals: int
    houses: int
    tech: int
    rp: int                  # 研究进度（不显示）
    mood: int
    soil: float              # 土壤肥力（隐藏）

    @property
    def cap(self):
        return HOUSE_CAP * self.houses


def weather_mult(weather, canals):
    if weather == "drought":
        return min(1.0, 0.5 + 0.15 * canals)      # 水渠缓解干旱（所有世界相同的公开规律）
    return 1.25 if weather == "bountiful" else 1.0


def harvest_value(farms, pop, canals, tech, soil, weather, y0, irr_gain, noise=1.0):
    """收成公式。模拟器和理想观察者共用，保证一致。"""
    worked = min(farms, pop / PEOPLE_PER_FARM)       # 劳动力不足时农田会闲置
    return round(worked * y0 * (1 + irr_gain * canals) * soil * weather_mult(weather, canals)
                 * (1 + 0.08 * tech) * noise)


def next_soil(soil, canals, irrigation):
    if irrigation == "salinize" and canals > 0:
        return max(SOIL_MIN, soil - SALT_PER_CANAL * canals)
    return min(1.0, soil + SOIL_RECOVER)


def step(P: Params, s: State, act: str, weather: str, noise: float = 1.0):
    """执行一年。返回 (新状态, 结果记录)。不修改输入。"""
    s = copy.copy(s)
    o = {"act": act, "ok": True, "weather": weather, "trade_in": 0, "feast_cost": 0,
         "tech_up": False, "deaths": 0, "left": 0, "crowded": False}
    food0, pop0 = s.food, s.pop

    # 1. 行动
    if act in ("farm", "canal", "house"):
        limit = {"farm": 99, "canal": MAX_CANALS, "house": MAX_HOUSES}[act]
        attr = {"farm": "farms", "canal": "canals", "house": "houses"}[act]
        if s.wood >= COST[act] and getattr(s, attr) < limit:
            s.wood -= COST[act]
            setattr(s, attr, getattr(s, attr) + 1)
        else:
            o["ok"] = False
    elif act == "forest":
        s.wood += round(45 * (1 + 0.1 * s.tech))
    elif act == "trade":
        if s.wood >= COST["trade"]:
            s.wood -= COST["trade"]
            o["trade_in"] = round(COST["trade"] * P.trade_rate)
        else:
            o["ok"] = False
    elif act == "research":
        s.rp += 1
        if s.rp >= 2 + s.tech and s.tech < MAX_TECH:
            s.tech += 1
            s.rp = 0
            o["tech_up"] = True
    elif act == "feast":
        cost = round(0.15 * s.pop)
        if s.food >= cost:
            o["feast_cost"] = cost
            s.mood += 12
        else:
            o["ok"] = False
    elif act == "rest":
        s.mood += 4

    # 2. 收成（使用行动之后的农田/水渠数）
    o["harvest"] = harvest_value(s.farms, s.pop, s.canals, s.tech, s.soil, weather, P.y0, IRR_GAIN[P.irrigation], noise)
    s.soil = next_soil(s.soil, s.canals, P.irrigation)

    # 3. 粮食与人口
    food = s.food + o["harvest"] + o["trade_in"] - o["feast_cost"] - s.pop
    if food < 0:
        o["deaths"] = math.ceil(-food / 2)
        s.pop = max(5, s.pop - o["deaths"])
        food = 0
        s.mood -= 12
    else:
        food -= round(0.08 * food)               # 储存损耗
        g = 0.07 if food >= 0.5 * s.pop else 0.02
        s.pop += round(s.pop * g)
    s.food = food
    if s.pop > s.cap:
        s.pop = s.cap
        s.mood -= 5
        o["crowded"] = True

    # 4. 情绪、迁出、木材
    s.mood += round((60 - s.mood) * 0.15)
    s.mood = max(0, min(100, s.mood))
    if s.mood < 25:
        o["left"] = round(0.06 * s.pop)
        s.pop -= o["left"]
    s.wood += 8
    s.year += 1

    o["food_dir"] = "up" if s.food > food0 else ("down" if s.food < food0 else "same")
    o["pop_dir"] = "up" if s.pop > pop0 else ("down" if s.pop < pop0 else "same")
    o["food"], o["pop"] = s.food, s.pop
    return s, o


class Policy:
    """带随机性的启发式策略：每个文明有自己的"性格"（行动偏好），再根据处境调整。"""

    def __init__(self, rng):
        self.base = {a: rng.gammavariate(1.0, 1.0) + 0.15 for a in ACTIONS}
        self.base["canal"] += 0.6          # 保证灌溉经常出现，否则灌溉规则无从观察
        self.base["trade"] += 0.3

    def act(self, s: State, rng):
        w = dict(self.base)
        if s.food < 1.2 * s.pop:
            w["farm"] *= 3
            w["trade"] *= 2
        if s.wood < 30:
            w["forest"] *= 4
        if s.pop > 0.9 * s.cap:
            w["house"] *= 4
        if s.mood < 40:
            w["feast"] *= 3
            w["rest"] *= 2
        if s.canals >= MAX_CANALS:
            w["canal"] = 0.02
        if s.tech >= MAX_TECH:
            w["research"] = 0.05
        acts, ws = zip(*w.items())
        return rng.choices(acts, ws)[0]


def make_world(seed, rules=None, years=40):
    rng = random.Random(seed)
    rules = rules or rng.choice(ALL_COMBOS)
    P = Params(**rules, y0=BASE_YIELD * rng.uniform(0.85, 1.15),
               trade_rate=TRADE_RATE[rules["trade"]] * rng.uniform(0.85, 1.15))
    s0 = State(year=1, pop=rng.randint(80, 140), food=rng.randint(200, 400), wood=rng.randint(40, 120),
               farms=3, canals=0, houses=3, tech=0, rp=0, mood=60, soil=1.0)
    wr = random.Random(seed * 7919 + 17)           # 天气：独立的随机流，分叉时保持一致
    weather = []
    for _ in range(years + 20):
        u = wr.random()
        weather.append("drought" if u < P_DROUGHT[rules["climate"]] else ("bountiful" if u > 1 - P_BOUNTIFUL else "normal"))
    nr = random.Random(seed * 7919 + 29)
    noise = [min(1.25, max(0.75, nr.gauss(1.0, YIELD_NOISE))) for _ in range(years + 20)]
    return P, s0, weather, noise


def simulate(seed, rules=None, years=40):
    """生成一个文明的完整历史。years[i] = {"state": 年初状态, "out": 本年结果}"""
    P, s, weather, noise = make_world(seed, rules, years)
    prng = random.Random(seed * 104729 + 3)
    pol = Policy(prng)
    hist = []
    for i in range(years):
        a = pol.act(s, prng)
        s2, o = step(P, s, a, weather[i], noise[i])
        hist.append({"state": s, "out": o})
        s = s2
    return {"seed": seed, "params": P, "weather": weather, "noise": noise, "years": hist, "final": s}


def fork(ep, year_idx, alt_action, n_years=5, policy_seed=None):
    """反事实：从第 year_idx 年年初复制世界，改为执行 alt_action，之后用同样的天气继续 n_years 年。"""
    P, weather, noise = ep["params"], ep["weather"], ep["noise"]
    s = ep["years"][year_idx]["state"]
    prng = random.Random(policy_seed if policy_seed is not None else ep["seed"] * 31 + year_idx)
    pol = Policy(random.Random(ep["seed"] * 104729 + 3))     # 同一个文明性格
    hist = []
    for j in range(n_years):
        a = alt_action if j == 0 else pol.act(s, prng)
        s2, o = step(P, s, a, weather[year_idx + j], noise[year_idx + j])
        hist.append({"state": s, "out": o})
        s = s2
    return hist


if __name__ == "__main__":
    from collections import Counter
    for combo in ALL_COMBOS[:3]:
        ep = simulate(1, combo, 40)
        print(combo, "y0=%.1f" % ep["params"].y0)
        for y in ep["years"][:12]:
            st, o = y["state"], y["out"]
            print(f"  Y{st.year:2d} pop {st.pop:3d} food {st.food:4d} wood {st.wood:3d} farms {st.farms} canals {st.canals} "
                  f"soil {st.soil:.2f} | {o['act']:8s}{'' if o['ok'] else '(x)'} {o['weather']:9s} harvest {o['harvest']}")
    acts = Counter()
    pops = []
    for sd in range(300):
        ep = simulate(sd, None, 40)
        acts.update(y["out"]["act"] for y in ep["years"])
        pops.append(ep["final"].pop)
    print("action freq", acts.most_common())
    print("final pop min/median/max", min(pops), sorted(pops)[150], max(pops))


In [ ]:
%%writefile /content/civlm/chronicle.py
"""
CivChronicle v0：把模拟器的历史写成文本
=======================================
两种文体，信息量完全相同，便于做"语言起了什么作用"的消融实验：
  structured  紧凑的 DSL 行
  narrative   英文叙事，每类事件有多种说法（防止模型只记模板）

每一年分成 4 段：state（年初状态）/ act（行动）/ weather（天气）/ outcome（结果）。
评测时可以在任意一段之后截断，让模型续写。
"""
import random

from engine import ACTIONS, WEATHERS

NAMES = ["Arvel", "Brannoch", "Celedor", "Dunmar", "Eskara", "Fennick", "Galdur", "Hollin", "Iskarn", "Jorvel",
         "Kestrel", "Lunmoor", "Marrow", "Nethis", "Orlan", "Pellam", "Quarry", "Rennick", "Sorrow", "Tamsin",
         "Ulvar", "Vessa", "Wyndham", "Yarrow", "Zennor"]
# 地貌描述只是"风味"，与隐藏规则无关：模型不能靠看名字猜规则
FLAVORS = ["a wide valley", "a stretch of low hills", "a plain beside a slow river", "the edge of an old forest",
           "a high plateau", "a shallow basin", "a windy coast", "a land of scattered lakes"]


def pl(n, word, plural=None):
    return f"{n} {word if n == 1 else (plural or word + 's')}"


# ---------------- structured ----------------
def s_header(ep):
    r = random.Random(ep["seed"])
    return f"<world> {r.choice(NAMES)}{ep['seed'] % 1000} | {r.choice(FLAVORS)}\n"


def s_state(st):
    return (f"Y{st.year} pop {st.pop} food {st.food} wood {st.wood} farms {st.farms} canals {st.canals} "
            f"cap {st.cap} tech {st.tech} mood {st.mood}\n")


def s_act(o):
    return f"act {o['act']}{'' if o['ok'] else ' failed'}\n"


def s_weather(o):
    return f"weather {o['weather']}\n"


def s_outcome(o):
    parts = [f"harvest {o['harvest']}"]
    if o["trade_in"]:
        parts.append(f"trade +{o['trade_in']}")
    if o["feast_cost"]:
        parts.append(f"feast -{o['feast_cost']}")
    if o["tech_up"]:
        parts.append("techup")
    if o["deaths"]:
        parts.append(f"starved {o['deaths']}")
    if o["left"]:
        parts.append(f"left {o['left']}")
    if o["crowded"]:
        parts.append("crowded")
    parts.append(f"food {o['food']} {o['food_dir']}")
    parts.append(f"pop {o['pop']} {o['pop_dir']}")
    return " ".join(parts) + "\n\n"


# ---------------- narrative ----------------
N_HEADER = [
    "This is the chronicle of {name}, a settlement in {flavor}.\n\n",
    "The people of {name} made their home in {flavor}. Here is their history.\n\n",
    "A chronicle of {name}, founded in {flavor}.\n\n",
]
N_STATE = [
    "Year {y}. {pop} people lived in {name}, with {food} measures of grain and {wood} loads of timber in store. "
    "They worked {farms} and {canals}, had room for {cap} people, knowledge level {tech}, and their mood stood at {mood}.",
    "In year {y}, {name} numbered {pop} souls. The granaries held {food} measures and the woodpiles {wood} loads; "
    "there were {farms}, {canals}, housing for {cap}, knowledge level {tech} and mood {mood}.",
    "Year {y} began with {pop} people, {food} grain, {wood} timber, {farms}, {canals}, space for {cap}, "
    "knowledge level {tech} and mood {mood}.",
]
N_ACT = {
    "farm": ["The council ordered new fields cleared for a farm.", "The elders decided to open another farm."],
    "canal": ["The council ordered a new canal dug.", "The elders set the people to digging an irrigation canal."],
    "forest": ["Woodcutters were sent into the forest.", "The council sent crews to fell timber."],
    "trade": ["A caravan of timber was sent to the traders.", "The elders offered timber to passing merchants."],
    "research": ["The scholars were given leave to study.", "The council invested in learning."],
    "feast": ["The council declared a great feast.", "The elders held a festival for the people."],
    "house": ["The council ordered new houses built.", "Builders raised new homes."],
    "rest": ["The people were allowed a quiet year.", "The council asked little of the people this year."],
}
N_FAIL = ["But there was not enough to carry it out.", "But the plan came to nothing."]
N_WEATHER = {
    "normal": ["The seasons passed without trouble.", "The rains came as usual.", "It was an ordinary year for weather."],
    "drought": ["A drought parched the fields.", "The rains failed and the land dried out.", "Drought struck the land."],
    "bountiful": ["The rains were generous.", "Gentle rains blessed the fields.", "The weather was kind to the farmers."],
}
N_HARVEST = ["The harvest brought in {h} measures of grain.", "The fields yielded {h} measures.",
             "{h} measures of grain were gathered."]
N_TRADE = ["The traders gave {g} measures of grain for the timber.", "In exchange, merchants left {g} measures of grain."]
N_FEAST = ["The feast consumed {c} measures.", "Feasting used up {c} measures of grain."]
N_TECH = ["The scholars reached a new level of knowledge.", "Learning advanced to a new level."]
N_DEATH = ["Hunger killed {d} people.", "{d} people died of hunger."]
N_LEFT = ["{l} people left in discontent.", "Unhappy, {l} people abandoned the settlement."]
N_CROWD = ["Crowding soured the mood.", "There was not enough room for everyone."]
# 粮食方向词与人口方向词不重叠，方便评测时解析
N_FOOD = {"up": ["The stores rose to {f} measures.", "The granaries climbed to {f} measures."],
          "down": ["The stores fell to {f} measures.", "The granaries sank to {f} measures."],
          "same": ["The stores held at {f} measures."]}
N_POP = {"up": ["The population grew to {p}.", "Their number increased to {p}."],
         "down": ["The population shrank to {p}.", "Their number dropped to {p}."],
         "same": ["The population stayed at {p}."]}
FOOD_DIR_WORDS = {"rose": "up", "climbed": "up", "fell": "down", "sank": "down", "held": "same",
                  "up": "up", "down": "down", "same": "same"}


def n_header(ep, rng):
    r = random.Random(ep["seed"])
    return rng.choice(N_HEADER).format(name=r.choice(NAMES), flavor=r.choice(FLAVORS))


def n_state(st, rng, name):
    return rng.choice(N_STATE).format(y=st.year, pop=st.pop, name=name, food=st.food, wood=st.wood,
                                      farms=pl(st.farms, "farm"), canals=pl(st.canals, "canal"),
                                      cap=st.cap, tech=st.tech, mood=st.mood)


def n_act(o, rng):
    t = " " + rng.choice(N_ACT[o["act"]])
    if not o["ok"]:
        t += " " + rng.choice(N_FAIL)
    return t


def n_weather(o, rng):
    return " " + rng.choice(N_WEATHER[o["weather"]])


def n_outcome(o, rng):
    p = [rng.choice(N_HARVEST).format(h=o["harvest"])]
    if o["trade_in"]:
        p.append(rng.choice(N_TRADE).format(g=o["trade_in"]))
    if o["feast_cost"]:
        p.append(rng.choice(N_FEAST).format(c=o["feast_cost"]))
    if o["tech_up"]:
        p.append(rng.choice(N_TECH))
    if o["deaths"]:
        p.append(rng.choice(N_DEATH).format(d=o["deaths"]))
    if o["left"]:
        p.append(rng.choice(N_LEFT).format(l=o["left"]))
    if o["crowded"]:
        p.append(rng.choice(N_CROWD))
    p.append(rng.choice(N_FOOD[o["food_dir"]]).format(f=o["food"]))
    p.append(rng.choice(N_POP[o["pop_dir"]]).format(p=o["pop"]))
    return " " + " ".join(p) + "\n\n"


# ---------------- 统一接口 ----------------
class Renderer:
    def __init__(self, ep, style, rng_seed=None):
        self.ep, self.style = ep, style
        self.rng = random.Random(rng_seed if rng_seed is not None else ep["seed"] * 13 + 5)
        self.name = random.Random(ep["seed"]).choice(NAMES)

    def header(self):
        return s_header(self.ep) if self.style == "structured" else n_header(self.ep, self.rng)

    def year(self, y):
        """返回 [state, act, weather, outcome] 四段文本，直接拼接即为该年全文。"""
        st, o = y["state"], y["out"]
        if self.style == "structured":
            return [s_state(st), s_act(o), s_weather(o), s_outcome(o)]
        return [n_state(st, self.rng, self.name), n_act(o, self.rng), n_weather(o, self.rng), n_outcome(o, self.rng)]

    def weather_candidates(self, act_seg_end=True):
        """天气的所有可能说法，按类别分组（用于评测时计算 P(天气类别)）。"""
        if self.style == "structured":
            return {w: [f"weather {w}\n"] for w in WEATHERS}
        return {w: [" " + t for t in N_WEATHER[w]] for w in WEATHERS}

    def whatif(self, year, act):
        if self.style == "structured":
            return f"whatif Y{year} act {act}\n"
        return f"What if: in year {year} the council had chosen differently. " \
               f"{N_ACT[act][0]}\n\n"


def render_episode(ep, style, cf=None):
    """完整文档。cf = (year_idx, alt_action, branch_years) 时，在主历史后附一段反事实分支。"""
    r = Renderer(ep, style)
    out = [r.header()]
    for y in ep["years"]:
        out += r.year(y)
    if cf is not None:
        idx, alt, branch = cf
        out.append(r.whatif(ep["years"][idx]["state"].year, alt))
        for y in branch:
            out += r.year(y)
    return "".join(out)


if __name__ == "__main__":
    from engine import simulate
    ep = simulate(3, None, 4)
    print(render_episode(ep, "structured"))
    print(render_episode(ep, "narrative"))


In [ ]:
%%writefile /content/civlm/tokenizer.py
"""
字节级 BPE 分词器（改自 energy_llm_v2/tokenizer.py）
- 数字按单个数位切分：118 → 1 1 8，模型才能学到数值之间的关系
- 词表 2048：文明史文本的词汇很有限，小词表让嵌入层更省参数
"""
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

SPECIAL_TOKENS = ["<|endoftext|>"]


def train_bpe(text_iterator, vocab_size, save_path):
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Digits(individual_digits=True),
        pre_tokenizers.ByteLevel(add_prefix_space=False),
    ])
    tk.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=2, special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
    tk.train_from_iterator(text_iterator, trainer)
    tk.save(save_path)
    return BPETokenizer(save_path)


class BPETokenizer:
    def __init__(self, path):
        self.tk = Tokenizer.from_file(path)
        self.eot_id = self.tk.token_to_id("<|endoftext|>")

    @property
    def vocab_size(self):
        return self.tk.get_vocab_size()

    def encode(self, text):
        return self.tk.encode(text, add_special_tokens=False).ids

    def encode_batch(self, texts):
        return [e.ids for e in self.tk.encode_batch(texts, add_special_tokens=False)]

    def decode(self, ids):
        return self.tk.decode(list(ids), skip_special_tokens=True)


In [ ]:
%%writefile /content/civlm/gen_data.py
"""
生成训练数据
============
1. 用 CivEngine 模拟 N 个文明（规则组合只来自 TRAIN_COMBOS，两种组合完全留给测试）
2. 用 CivChronicle 写成文本：叙事 / 结构化按比例混合，部分文档附反事实分支
3. 训练字节级 BPE 分词器（数字按单个数位切分），编码成 train.bin / val.bin
4. 写出评测世界清单 eval_worlds.json：
     id   训练见过的规则组合、但没见过的种子
     ood  训练从未出现的规则组合

用法：
  python gen_data.py                       # 默认 2 万个文明，约 5000 万 token
  python gen_data.py --n_worlds 2000       # 快速试跑
  python gen_data.py --narrative_frac 0    # 消融：只用结构化文本
"""
import argparse
import json
import os
import random
import time
from multiprocessing import Pool

import numpy as np

from chronicle import render_episode
from engine import ACTIONS, ALL_COMBOS, HELDOUT_COMBOS, TRAIN_COMBOS, fork, simulate
from tokenizer import train_bpe

VAL_SEED0, EVAL_SEED0 = 10_000_000, 20_000_000


def make_doc(job):
    seed, years, narrative_frac, cf_frac = job
    r = random.Random(seed * 2654435761 % (2 ** 31))
    rules = r.choice(TRAIN_COMBOS)
    ep = simulate(seed, rules, years)
    style = "narrative" if r.random() < narrative_frac else "structured"
    cf = None
    if r.random() < cf_frac:
        idx = r.randrange(5, years - 6)
        alt = r.choice([a for a in ACTIONS if a != ep["years"][idx]["out"]["act"]])
        cf = (idx, alt, fork(ep, idx, alt, n_years=5))
    return render_episode(ep, style, cf)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_dir", default="data")
    ap.add_argument("--n_worlds", type=int, default=20000)
    ap.add_argument("--n_val", type=int, default=500)
    ap.add_argument("--n_eval", type=int, default=200, help="每个评测集（id/ood）的世界数")
    ap.add_argument("--years", type=int, default=40)
    ap.add_argument("--narrative_frac", type=float, default=0.5)
    ap.add_argument("--cf_frac", type=float, default=0.1, help="附带反事实分支的文档比例")
    ap.add_argument("--vocab_size", type=int, default=2048)
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 1))
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    t0 = time.time()

    jobs = [(s, args.years, args.narrative_frac, args.cf_frac) for s in range(args.n_worlds)]
    vjobs = [(VAL_SEED0 + s, args.years, args.narrative_frac, args.cf_frac) for s in range(args.n_val)]
    with Pool(args.workers) as pool:
        train_docs = pool.map(make_doc, jobs, chunksize=64)
        val_docs = pool.map(make_doc, vjobs, chunksize=64)
    print(f"生成 {len(train_docs)} 篇训练文档、{len(val_docs)} 篇验证文档，用时 {time.time() - t0:.0f}s")
    print("示例：\n" + train_docs[0][:600] + "\n...")

    tok_path = os.path.join(args.out_dir, "tokenizer.json")
    sample = random.Random(0).sample(train_docs, min(3000, len(train_docs)))
    tok = train_bpe(iter(sample), args.vocab_size, tok_path)
    print(f"分词器词表 {tok.vocab_size}")

    def write_bin(docs, path):
        n = 0
        with open(path, "wb") as f:
            for i in range(0, len(docs), 1000):
                ids = []
                for e in tok.encode_batch(docs[i:i + 1000]):
                    ids += e + [tok.eot_id]
                arr = np.array(ids, dtype=np.uint16)
                arr.tofile(f)
                n += len(arr)
        return n

    ntr = write_bin(train_docs, os.path.join(args.out_dir, "train.bin"))
    nva = write_bin(val_docs, os.path.join(args.out_dir, "val.bin"))

    er = random.Random(1)
    eval_worlds = {
        "id": [{"seed": EVAL_SEED0 + i, "rules": er.choice(TRAIN_COMBOS)} for i in range(args.n_eval)],
        "ood": [{"seed": EVAL_SEED0 + 100000 + i, "rules": HELDOUT_COMBOS[i % len(HELDOUT_COMBOS)]}
                for i in range(args.n_eval)],
        "years": args.years,
    }
    json.dump(eval_worlds, open(os.path.join(args.out_dir, "eval_worlds.json"), "w"), indent=1)
    meta = {"train_tokens": ntr, "val_tokens": nva, "n_worlds": args.n_worlds, "years": args.years,
            "narrative_frac": args.narrative_frac, "cf_frac": args.cf_frac, "vocab_size": tok.vocab_size,
            "train_combos": TRAIN_COMBOS, "heldout_combos": HELDOUT_COMBOS,
            "tokens_per_doc": ntr / len(train_docs)}
    json.dump(meta, open(os.path.join(args.out_dir, "meta.json"), "w"), indent=1)
    print(f"train {ntr / 1e6:.1f}M tokens，val {nva / 1e6:.2f}M tokens，平均每篇 {ntr / len(train_docs):.0f} tokens")
    print(f"完成，用时 {time.time() - t0:.0f}s → {args.out_dir}/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/civlm/observer.py
"""
理想观察者（Ideal Bayesian Observer）—— 评测的上限参照线
=========================================================
它知道所有规则族（3 种灌溉 × 2 种气候 × 2 种贸易）和抖动范围，
只用上下文窗口里那 k 年的观测（收成、贸易所得、天气）做贝叶斯推断，然后预测下一年。

说明：为了计算土壤盐碱化，它可以看到窗口之前的水渠修建记录（比语言模型多一点信息），
因此它是"偏强"的上限。语言模型越接近它，说明从上下文推断隐藏规律的能力越强。

三个隐藏规律相互独立，分别推断：
  收成  ← (灌溉规则, 基础产量抖动)
  贸易  ← (贸易规则, 汇率抖动)
  天气  ← 气候规则
"""
import copy
import math

import numpy as np

from engine import (BASE_YIELD, IRR_GAIN, P_BOUNTIFUL, P_DROUGHT, RULES, TRADE_RATE, YIELD_NOISE, Params,
                    harvest_value, next_soil, step)

Y0_GRID = np.linspace(0.85, 1.15, 31) * BASE_YIELD
TR_GRID = np.linspace(0.85, 1.15, 31)
SIGMA_LOG = YIELD_NOISE   # 收成有约 10% 的随机波动（观察者知道波动幅度，但不知道具体值）
SIGMA_TRADE = 0.02        # 贸易所得没有随机波动，只有取整误差


def soil_series(years, irrigation):
    """按水渠历史算出每年年初（收成时）的土壤肥力。"""
    soil, out = 1.0, []
    for y in years:
        out.append(soil)
        st, o = y["state"], y["out"]
        canals = st.canals + (1 if (o["act"] == "canal" and o["ok"]) else 0)
        soil = next_soil(soil, canals, irrigation)
    return out


def post_action(st, o):
    s = copy.copy(st)
    if o["ok"] and o["act"] in ("farm", "canal", "house"):
        attr = {"farm": "farms", "canal": "canals", "house": "houses"}[o["act"]]
        setattr(s, attr, getattr(s, attr) + 1)
    if o["ok"] and o["act"] == "research" and o["tech_up"]:
        s.tech += 1
    return s


class Observer:
    def __init__(self, ep):
        self.ep = ep
        self.years = ep["years"]
        self.soils = {irr: soil_series(self.years, irr) for irr in RULES["irrigation"]}

    def _harvest(self, i, irr, y0):
        st, o = self.years[i]["state"], self.years[i]["out"]
        s = post_action(st, o)
        return harvest_value(s.farms, s.pop, s.canals, s.tech, self.soils[irr][i], o["weather"], y0, IRR_GAIN[irr])

    def posterior(self, window):
        """window: 可见年份的下标列表。返回各隐藏变量的后验。"""
        # 收成：灌溉 × y0
        hyps, logw = [], []
        for irr in RULES["irrigation"]:
            for y0 in Y0_GRID:
                lw = 0.0
                for i in window:
                    pred = max(1, self._harvest(i, irr, y0))
                    obs = max(1, self.years[i]["out"]["harvest"])
                    lw += -0.5 * ((math.log(obs) - math.log(pred)) / SIGMA_LOG) ** 2
                hyps.append((irr, y0))
                logw.append(lw)
        w = np.exp(np.array(logw) - max(logw))
        w /= w.sum()
        # 贸易
        th, tl = [], []
        for tr in RULES["trade"]:
            for j in TR_GRID:
                rate = TRADE_RATE[tr] * j
                lw = 0.0
                for i in window:
                    o = self.years[i]["out"]
                    if o["trade_in"]:
                        lw += -0.5 * ((math.log(o["trade_in"]) - math.log(round(30 * rate))) / SIGMA_TRADE) ** 2
                th.append((tr, rate))
                tl.append(lw)
        tw = np.exp(np.array(tl) - max(tl))
        tw /= tw.sum()
        # 气候
        cl = {}
        for c in RULES["climate"]:
            pd, pb = P_DROUGHT[c], P_BOUNTIFUL
            lw = 0.0
            for i in window:
                wx = self.years[i]["out"]["weather"]
                lw += math.log(pd if wx == "drought" else (pb if wx == "bountiful" else 1 - pd - pb))
            cl[c] = lw
        m = max(cl.values())
        z = sum(math.exp(v - m) for v in cl.values())
        cpost = {c: math.exp(v - m) / z for c, v in cl.items()}
        return hyps, w, th, tw, cpost

    def predict(self, window, t):
        """预测第 t 年（下标）的收成、粮食方向和干旱概率。"""
        hyps, w, th, tw, cpost = self.posterior(window)
        st, o = self.years[t]["state"], self.years[t]["out"]
        rate = float(np.dot(tw, [r for _, r in th]))
        logs, dirs = [], {"up": 0.0, "down": 0.0, "same": 0.0}
        top = np.argsort(-w)[:60]                 # 只算权重最大的若干假设，足够精确
        tot = w[top].sum()
        for k in top:
            irr, y0 = hyps[k]
            P = Params(irrigation=irr, climate="wet", trade="rich", y0=y0, trade_rate=rate)
            s = copy.copy(st)
            s.soil = self.soils[irr][t]
            _, oo = step(P, s, o["act"], o["weather"])
            logs.append((math.log(max(1, oo["harvest"])), w[k] / tot))
            dirs[oo["food_dir"]] += w[k] / tot
        logs.sort()
        acc, med = 0.0, logs[-1][0]
        for v, ww in logs:                         # 加权中位数（最小化对数误差）
            acc += ww
            if acc >= 0.5:
                med = v
                break
        p_drought = sum(cpost[c] * P_DROUGHT[c] for c in cpost)
        return {"harvest": math.exp(med), "food_dir": max(dirs, key=dirs.get), "p_drought": p_drought,
                "p_irrigation": {irr: float(sum(w[i] for i, h in enumerate(hyps) if h[0] == irr))
                                 for irr in RULES["irrigation"]}}


In [ ]:
%%writefile /content/civlm/model.py
"""
LLaMA 风格的迷你语言模型（约 2600 万参数）
==========================================
与 v1 的 GPT-2 结构相比，采用了现代大模型（LLaMA / Qwen）的标准组件：
  - RMSNorm      代替 LayerNorm（更简单，没有均值和偏置）
  - RoPE         旋转位置编码，代替可学习的位置嵌入
  - SwiGLU       门控前馈网络，代替 GELU-MLP
  - KV Cache     生成时缓存历史的 K/V，避免重复计算
"""
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    vocab_size: int = 6400
    block_size: int = 512
    n_layer: int = 8
    n_head: int = 8
    n_embd: int = 512
    dropout: float = 0.0
    rope_theta: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight


def precompute_rope(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)          # (T, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, nh, T, hd)。把向量两两一组看成复数，按位置旋转一个角度。"""
    d = x.size(-1) // 2
    x1, x2 = x[..., :d], x[..., d:]
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh = cfg.n_head
        self.hd = cfg.n_embd // cfg.n_head
        self.wq = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.wk = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.wv = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.wo = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin, past=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:                       # 拼上缓存的历史 K/V
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        # 无缓存（训练/首次前向）时用因果掩码；有缓存时新 token 可以看到全部历史
        y = F.scaled_dot_product_attention(q, k, v, is_causal=past is None and T > 1,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.wo(y), (k, v)


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        hidden = int(8 * cfg.n_embd / 3)
        hidden = 64 * ((hidden + 63) // 64)
        self.w1 = nn.Linear(cfg.n_embd, hidden, bias=False)   # 门控
        self.w3 = nn.Linear(cfg.n_embd, hidden, bias=False)   # 值
        self.w2 = nn.Linear(hidden, cfg.n_embd, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.n_embd)
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg.n_embd)
        self.ffn = SwiGLU(cfg)

    def forward(self, x, cos, sin, past=None):
        h, kv = self.attn(self.norm1(x), cos, sin, past)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x, kv


class MiniLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight     # 输入输出嵌入共享
        cos, sin = precompute_rope(cfg.n_embd // cfg.n_head, cfg.block_size, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        for n, p in self.named_parameters():
            if n.endswith("wo.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layer))

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None, past_kv=None, start_pos=0):
        B, T = idx.shape
        assert start_pos + T <= self.cfg.block_size, "超出最大上下文长度"
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]
        x = self.drop(self.tok_emb(idx))
        new_kv = []
        for i, blk in enumerate(self.blocks):
            x, kv = blk(x, cos, sin, past_kv[i] if past_kv is not None else None)
            new_kv.append(kv)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        return logits, loss, new_kv

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=None, top_p=None,
                 eos_id=None, repetition_penalty=1.0):
        """
        idx: (B, T) 等长提示。temperature=0 为贪心解码。
        返回 (B, T + 新生成长度)；遇到 eos 的序列之后用 eos 填充。
        """
        B = idx.size(0)
        finished = torch.zeros(B, dtype=torch.bool, device=idx.device)
        max_new_tokens = min(max_new_tokens, self.cfg.block_size - idx.size(1))
        logits, _, past = self(idx)                         # 预填充（prefill）
        logits = logits[:, -1, :].float()
        out = idx
        for step in range(max_new_tokens):
            if repetition_penalty != 1.0:                   # 抑制重复（小模型常见问题）
                score = torch.gather(logits, 1, out)
                score = torch.where(score > 0, score / repetition_penalty, score * repetition_penalty)
                logits = logits.scatter(1, out, score)
            if temperature == 0:
                nxt = logits.argmax(-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("inf")
                probs = F.softmax(logits, dim=-1)
                if top_p and top_p < 1.0:                    # nucleus 采样
                    sp, si = torch.sort(probs, descending=True)
                    cum = sp.cumsum(-1)
                    sp[(cum - sp) > top_p] = 0
                    probs = torch.zeros_like(probs).scatter(1, si, sp)
                    probs = probs / probs.sum(-1, keepdim=True)
                nxt = torch.multinomial(probs, 1)
            if eos_id is not None:
                nxt = torch.where(finished.unsqueeze(1), torch.full_like(nxt, eos_id), nxt)
                finished |= nxt.squeeze(1) == eos_id
            out = torch.cat([out, nxt], dim=1)
            if (eos_id is not None and finished.all()) or step == max_new_tokens - 1:
                break
            logits, _, past = self(nxt, past_kv=past, start_pos=out.size(1) - 1)
            logits = logits[:, -1, :].float()
        return out


def save_checkpoint(model, path, extra=None):
    torch.save({"model": model.state_dict(), "config": asdict(model.cfg), "extra": extra or {}}, path)


def load_checkpoint(path, device="cpu"):
    ck = torch.load(path, map_location=device)
    m = MiniLM(ModelConfig(**ck["config"]))
    m.load_state_dict(ck["model"])
    return m.to(device)


In [ ]:
%%writefile /content/civlm/utils.py
"""通用工具：设备、混合精度、随机种子（同 energy_llm_v2/utils.py）"""
import math
import random
import sys
from contextlib import nullcontext

import numpy as np
import torch

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")   # Windows 控制台中文
except Exception:
    pass


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device(name=None):
    if name:
        return name
    return "cuda" if torch.cuda.is_available() else "cpu"


def amp_setup(device):
    """2080Ti（算力 7.5）不支持 bf16 → fp16 + GradScaler；30 系及以上 → bf16；CPU → 全精度"""
    if device.startswith("cuda"):
        major, _ = torch.cuda.get_device_capability()
        dtype = torch.bfloat16 if major >= 8 else torch.float16
        ctx = torch.autocast(device_type="cuda", dtype=dtype)
        scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16))
        print(f"[AMP] GPU 算力 {major}.x，使用 {dtype}")
        return ctx, scaler
    return nullcontext(), torch.amp.GradScaler("cpu", enabled=False)


def cosine_lr(step, max_steps, base_lr, min_lr, warmup):
    if step < warmup:
        return base_lr * (step + 1) / warmup
    ratio = min(1.0, (step - warmup) / max(1, max_steps - warmup))
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * ratio))


def print_gpu_mem():
    if torch.cuda.is_available():
        print(f"[显存] 峰值占用 {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GB")


In [ ]:
%%writefile /content/civlm/pretrain.py
"""
从零预训练 CivLM（改自 energy_llm_v2/pretrain.py）
==================================================
只做下一个 token 预测，不告诉模型任何世界规律。

默认 8 层 × 512 维 ≈ 2600 万参数；上下文 2048（叙事文体 12 年历史约 1600 token）。
batch 16 × 2048 = 每步 3.3 万 token。2080Ti 预计每步 0.6~1 秒（估算，以屏幕显示为准）。

每 --ckpt_every 步另存一个检查点到 out/<name>/ckpts/，用于实验 1（能力随训练形成的曲线）：
  python eval_icl.py --ckpt_dir out/civlm26m/ckpts

100M 配置示例（先确认 26M 跑通、曲线有意义再上）：
  python pretrain.py --name civlm100m --n_layer 12 --n_head 12 --n_embd 768 --batch_size 8 --grad_accum 2
"""
import argparse
import os
import time

import numpy as np
import torch

from chronicle import Renderer
from engine import HELDOUT_COMBOS, simulate
from model import MiniLM, ModelConfig, load_checkpoint, save_checkpoint
from tokenizer import BPETokenizer
from utils import amp_setup, cosine_lr, get_device, print_gpu_mem, set_seed


def sample_prompts():
    """取一个没见过的世界，给 3 年历史，让模型续写第 4 年的结果。"""
    ep = simulate(99_999_999, HELDOUT_COMBOS[0], 6)
    out = []
    for style in ("structured", "narrative"):
        r = Renderer(ep, style)
        segs = [r.year(y) for y in ep["years"]]
        p = r.header() + "".join("".join(s) for s in segs[:3]) + "".join(segs[3][:3])
        out.append((p, "".join(segs[3][3:]).strip()))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--name", default="civlm26m")
    ap.add_argument("--n_layer", type=int, default=8)
    ap.add_argument("--n_head", type=int, default=8)
    ap.add_argument("--n_embd", type=int, default=512)
    ap.add_argument("--block_size", type=int, default=2048)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--grad_accum", type=int, default=1)
    ap.add_argument("--max_steps", type=int, default=5000)
    ap.add_argument("--lr", type=float, default=6e-4)
    ap.add_argument("--min_lr", type=float, default=6e-5)
    ap.add_argument("--warmup", type=int, default=300)
    ap.add_argument("--weight_decay", type=float, default=0.1)
    ap.add_argument("--eval_interval", type=int, default=250)
    ap.add_argument("--ckpt_every", type=int, default=500)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--device", default=None)
    ap.add_argument("--seed", type=int, default=1337)
    args = ap.parse_args()

    set_seed(args.seed)
    device = get_device(args.device)
    out_dir = os.path.join("out", args.name)
    ck_dir = os.path.join(out_dir, "ckpts")
    os.makedirs(ck_dir, exist_ok=True)
    best_path = os.path.join(out_dir, "best.pt")

    tok = BPETokenizer(os.path.join(args.data_dir, "tokenizer.json"))
    train = np.memmap(os.path.join(args.data_dir, "train.bin"), dtype=np.uint16, mode="r")
    val = np.memmap(os.path.join(args.data_dir, "val.bin"), dtype=np.uint16, mode="r")
    if len(val) < args.block_size + 2:
        val = train
    tok_per_step = args.batch_size * args.grad_accum * args.block_size
    print(f"训练集 {len(train) / 1e6:.1f}M tokens；计划训练 {args.max_steps * tok_per_step / 1e6:.0f}M tokens"
          f"（约 {args.max_steps * tok_per_step / len(train):.1f} 轮）")

    def get_batch(d):
        ix = np.random.randint(0, len(d) - args.block_size - 1, size=args.batch_size)
        x = np.stack([d[i:i + args.block_size] for i in ix]).astype(np.int64)
        y = np.stack([d[i + 1:i + 1 + args.block_size] for i in ix]).astype(np.int64)
        return torch.from_numpy(x).to(device, non_blocking=True), torch.from_numpy(y).to(device, non_blocking=True)

    if args.resume and os.path.exists(best_path):
        model = load_checkpoint(best_path, device)
        print(f"从 {best_path} 继续训练")
    else:
        cfg = ModelConfig(vocab_size=tok.vocab_size, block_size=args.block_size,
                          n_layer=args.n_layer, n_head=args.n_head, n_embd=args.n_embd)
        model = MiniLM(cfg).to(device)
    print(f"参数量 {model.num_params() / 1e6:.1f}M，设备 {device}")

    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": args.weight_decay},
                             {"params": no_decay, "weight_decay": 0.0}], lr=args.lr, betas=(0.9, 0.95))
    amp_ctx, scaler = amp_setup(device)
    prompts = sample_prompts()

    @torch.no_grad()
    def evaluate():
        model.eval()
        losses = []
        for _ in range(20):
            x, y = get_batch(val)
            with amp_ctx:
                losses.append(model(x, y)[1].item())
        for p, truth in prompts:
            ids = torch.tensor([tok.encode(p)], device=device)
            with amp_ctx:
                out = model.generate(ids, 50, temperature=0, eos_id=tok.eot_id)
            gen = tok.decode(out[0, ids.size(1):].tolist()).split("\n\n")[0].strip()
            print("  预测 →", gen.replace("\n", " | ")[:150])
            print("  真实 →", truth.replace("\n", " | ")[:150])
        model.train()
        return float(np.mean(losses))

    log = open(os.path.join(out_dir, "pretrain_log.csv"), "a" if args.resume else "w", encoding="utf-8")
    if not args.resume:
        log.write("step,tokens,train_loss,val_loss,lr\n")
    best, t0 = float("inf"), time.time()
    model.train()
    try:
        for step in range(args.max_steps + 1):
            lr = cosine_lr(step, args.max_steps, args.lr, args.min_lr, args.warmup)
            for g in opt.param_groups:
                g["lr"] = lr
            if step % args.eval_interval == 0 or step == args.max_steps:
                vl = evaluate()
                print(f"[step {step}] val_loss {vl:.4f}  已用 {(time.time() - t0) / 60:.1f} 分钟")
                log.write(f"{step},{step * tok_per_step},,{vl:.4f},{lr:.6f}\n")
                log.flush()
                if vl < best:
                    best = vl
                    save_checkpoint(model, best_path, {"step": step, "val_loss": vl})
            if step % args.ckpt_every == 0 or step == args.max_steps:
                save_checkpoint(model, os.path.join(ck_dir, f"step_{step:06d}.pt"),
                                {"step": step, "tokens": step * tok_per_step})
            if step == args.max_steps:
                break
            for _ in range(args.grad_accum):
                x, y = get_batch(train)
                with amp_ctx:
                    loss = model(x, y)[1] / args.grad_accum
                scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if step % 25 == 0:
                el = time.time() - t0
                eta = el / max(step, 1) * (args.max_steps - step) / 60
                print(f"step {step:6d} | loss {loss.item() * args.grad_accum:.4f} | lr {lr:.2e} | 剩余约 {eta:.0f} 分钟", end="\r")
                log.write(f"{step},{step * tok_per_step},{loss.item() * args.grad_accum:.4f},,{lr:.6f}\n")
    except KeyboardInterrupt:
        print("\n手动中断。")
    log.close()
    print(f"\n预训练结束，最佳 val_loss {best:.4f} → {best_path}")
    print_gpu_mem()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/civlm/eval_icl.py
"""
实验 2：上下文系统辨识 —— 给的历史越多，模型对没见过的世界预测得越准吗？
=========================================================================
对每个评测世界，选目标年 T，只给模型 T 之前的 k 年历史（k = 0,1,2,4,8,12），
再给第 T 年的年初状态、行动、天气，让模型续写结果。

指标：
  harvest_err   收成预测的 |log(预测/真实)|，越小越好（0.1 ≈ 差 10%）
  within10      收成误差在 10% 以内的比例
  food_acc      粮食涨/跌/持平方向的准确率
  drought_brier 在看到天气之前，模型给出的 P(干旱) 的 Brier 分数（越小越好）
  parse         输出能被解析的比例

参照线：
  persist   收成 = 上一年收成；粮食方向 = 上一年方向
  observer  理想贝叶斯观察者（知道所有规则族，见 observer.py）

两个评测集：
  id   训练见过的规则组合、新种子
  ood  训练中从未出现的规则组合

用法：
  python eval_icl.py --ckpt out/civlm26m/best.pt
  python eval_icl.py --ckpt_dir out/civlm26m/ckpts      # 实验 1：每个检查点都评一遍，画学习曲线
  python eval_icl.py --baselines_only                    # 不需要模型，只算两条参照线
"""
import argparse
import glob
import json
import math
import os
import random
import re
import time
from collections import defaultdict

import numpy as np

from chronicle import FOOD_DIR_WORDS, Renderer
from engine import simulate
from observer import Observer

DIR_RE = re.compile(r"\b(" + "|".join(FOOD_DIR_WORDS) + r")\b")
INT_RE = re.compile(r"\d+")
ERR_CAP = 1.0


def parse_outcome(text):
    m = INT_RE.search(text)
    if not m:
        return None, None
    harvest = int(m.group())
    d = DIR_RE.search(text, m.end())
    return harvest, (FOOD_DIR_WORDS[d.group(1)] if d else None)


def logerr(pred, true):
    if pred is None:
        return ERR_CAP
    return min(ERR_CAP, abs(math.log(max(1, pred) / max(1, true))))


def build_cases(worlds, years, ks, styles, targets_per_world):
    """预先构造全部评测样本（与模型无关），参照线也在这里算好。"""
    cases = []
    for w in worlds:
        ep = simulate(w["seed"], w["rules"], years)
        ob = Observer(ep)
        rng = random.Random(w["seed"])
        Ts = rng.sample(range(max(ks) + 1, years - 1), targets_per_world)
        renders = {st: Renderer(ep, st) for st in styles}
        segs = {st: [renders[st].year(y) for y in ep["years"]] for st in styles}
        for T in Ts:
            o = ep["years"][T]["out"]
            canals = ep["years"][T + 1]["state"].canals
            for k in ks:
                window = list(range(T - k, T))
                obs = ob.predict(window, T)
                last = ep["years"][T - 1]["out"] if k > 0 else None
                base = {"seed": w["seed"], "rules": w["rules"], "T": T, "k": k, "canals": canals,
                        "true_harvest": o["harvest"], "true_dir": o["food_dir"],
                        "true_drought": int(o["weather"] == "drought"),
                        "obs_harvest": obs["harvest"], "obs_dir": obs["food_dir"], "obs_pdrought": obs["p_drought"],
                        "obs_p_true_irr": obs["p_irrigation"][w["rules"]["irrigation"]],
                        "persist_harvest": last["harvest"] if last else None,
                        "persist_dir": last["food_dir"] if last else "up"}
                for st in styles:
                    r = renders[st]
                    hist = "".join("".join(segs[st][i]) for i in window)
                    cases.append(dict(base, style=st,
                                      header=r.header(),
                                      hist=hist,
                                      pre_weather="".join(segs[st][T][:2]),
                                      weather_seg=segs[st][T][2],
                                      weather_cands=r.weather_candidates()))
    return cases


# ---------------- 模型部分 ----------------
class LMScorer:
    def __init__(self, ckpt, device=None):
        import torch
        from model import load_checkpoint
        from tokenizer import BPETokenizer
        from utils import amp_setup, get_device
        self.torch = torch
        self.device = get_device(device)
        self.model = load_checkpoint(ckpt, self.device).eval()
        self.block = self.model.cfg.block_size
        data_dir = os.environ.get("CIVLM_DATA", "data")
        self.tok = BPETokenizer(os.path.join(data_dir, "tokenizer.json"))
        self.amp, _ = amp_setup(self.device)

    def _ids(self, header, body, reserve):
        h, b = self.tok.encode(header), self.tok.encode(body)
        room = self.block - reserve - len(h)
        if len(b) > room:                       # 太长时从左边截断历史，保留世界标题
            b = b[-room:]
        return h + b

    def run(self, case, max_new=48):
        torch = self.torch
        F = torch.nn.functional
        with torch.no_grad(), self.amp:
            # (a) 天气之前：P(干旱)
            ids = self._ids(case["header"], case["hist"] + case["pre_weather"], reserve=40)
            x = torch.tensor([ids], device=self.device)
            logits, _, past = self.model(x)
            last_lp = F.log_softmax(logits[0, -1].float(), -1)
            cls_lp = {}
            for wcls, texts in case["weather_cands"].items():
                lps = []
                for t in texts:
                    c = self.tok.encode(t)
                    lp = last_lp[c[0]].item()
                    if len(c) > 1:
                        lg, _, _ = self.model(torch.tensor([c[:-1]], device=self.device), past_kv=past,
                                              start_pos=len(ids))
                        lsm = F.log_softmax(lg[0].float(), -1)
                        lp += lsm[torch.arange(len(c) - 1), torch.tensor(c[1:], device=self.device)].sum().item()
                    lps.append(lp)
                m = max(lps)
                cls_lp[wcls] = m + math.log(sum(math.exp(v - m) for v in lps))
            m = max(cls_lp.values())
            z = sum(math.exp(v - m) for v in cls_lp.values())
            p_drought = math.exp(cls_lp["drought"] - m) / z
            # (b) 天气之后：续写结果
            ids = self._ids(case["header"], case["hist"] + case["pre_weather"] + case["weather_seg"], reserve=max_new)
            x = torch.tensor([ids], device=self.device)
            out = self.model.generate(x, max_new, temperature=0, eos_id=self.tok.eot_id)
            gen = self.tok.decode(out[0, len(ids):].tolist())
        gen = gen.split("\n\n")[0]
        h, d = parse_outcome(gen)
        return {"lm_harvest": h, "lm_dir": d, "lm_pdrought": p_drought, "lm_text": gen[:200]}


# ---------------- 汇总 ----------------
def summarize(cases, with_lm):
    groups = defaultdict(list)
    for c in cases:
        groups[(c["split"], c["style"], c["k"])].append(c)
    rows = []
    for (split, style, k), cs in sorted(groups.items()):
        r = {"split": split, "style": style, "k": k, "n": len(cs)}
        irr = [c for c in cs if c["canals"] > 0]
        for who in (["lm"] if with_lm else []) + ["obs", "persist"]:
            hk, dk = f"{who}_harvest", f"{who}_dir"
            if who == "persist" and k == 0:
                r.update({f"{who}_harvest_err": None, f"{who}_within10": None, f"{who}_food_acc": None,
                          f"{who}_harvest_err_irrigated": None})
                continue
            errs = [logerr(c[hk], c["true_harvest"]) for c in cs]
            r[f"{who}_harvest_err"] = float(np.mean(errs))
            r[f"{who}_within10"] = float(np.mean([e < math.log(1.1) for e in errs]))
            r[f"{who}_food_acc"] = float(np.mean([c[dk] == c["true_dir"] for c in cs]))
            r[f"{who}_harvest_err_irrigated"] = float(np.mean([logerr(c[hk], c["true_harvest"]) for c in irr])) if irr else None
        for who in (["lm"] if with_lm else []) + ["obs"]:
            r[f"{who}_drought_brier"] = float(np.mean([(c[f"{who}_pdrought"] - c["true_drought"]) ** 2 for c in cs]))
        if with_lm:
            r["lm_parse"] = float(np.mean([c["lm_harvest"] is not None for c in cs]))
        r["obs_p_true_irr"] = float(np.mean([c["obs_p_true_irr"] for c in cs]))
        rows.append(r)
    return rows


def fmt(v, w=6):
    return f"{v:{w}.3f}" if isinstance(v, float) else f"{'-':>{w}}"


def print_table(rows, with_lm):
    print("\n收成对数误差（越小越好）| 粮食方向准确率 | 干旱 Brier")
    head = f"{'split':5s} {'style':10s} {'k':>3s} | " + ("  LM   " if with_lm else "") + \
           " obs    persist | " + ("  LM   " if with_lm else "") + " obs    persist | " + \
           ("  LM   " if with_lm else "") + " obs   " + (" | parse" if with_lm else "")
    print(head)
    print("-" * len(head))
    for r in rows:
        line = f"{r['split']:5s} {r['style']:10s} {r['k']:3d} | "
        line += (fmt(r.get("lm_harvest_err")) + " " if with_lm else "") + fmt(r["obs_harvest_err"]) + " " + fmt(r["persist_harvest_err"]) + " | "
        line += (fmt(r.get("lm_food_acc")) + " " if with_lm else "") + fmt(r["obs_food_acc"]) + " " + fmt(r["persist_food_acc"]) + " | "
        line += (fmt(r.get("lm_drought_brier")) + " " if with_lm else "") + fmt(r["obs_drought_brier"])
        if with_lm:
            line += " | " + fmt(r["lm_parse"], 5)
        print(line)


def plot(rows, path, title, with_lm):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    splits = sorted({r["split"] for r in rows})
    styles = sorted({r["style"] for r in rows})
    fig, axes = plt.subplots(1, len(splits), figsize=(6 * len(splits), 4.2), squeeze=False)
    for ax, sp in zip(axes[0], splits):
        rs0 = [r for r in rows if r["split"] == sp and r["style"] == styles[0]]
        ks = [r["k"] for r in rs0]
        ax.plot(ks, [r["obs_harvest_err"] for r in rs0], "k--", label="ideal observer")
        pk = [(r["k"], r["persist_harvest_err"]) for r in rs0 if r["persist_harvest_err"] is not None]
        ax.plot([a for a, _ in pk], [b for _, b in pk], ":", color="gray", label="persistence")
        if with_lm:
            for st in styles:
                rs = [r for r in rows if r["split"] == sp and r["style"] == st]
                ax.plot([r["k"] for r in rs], [r["lm_harvest_err"] for r in rs], "o-", label=f"LM ({st})")
        ax.set_title(f"{title} · {sp}")
        ax.set_xlabel("years of history in context (k)")
        ax.set_ylabel("harvest |log error|")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    print(f"图 → {path}")


def evaluate_ckpt(ckpt, cases, args):
    scorer = LMScorer(ckpt, args.device)
    t0 = time.time()
    out = []
    for i, c in enumerate(cases):
        out.append(dict(c, **scorer.run(c)))
        if (i + 1) % 50 == 0:
            el = time.time() - t0
            print(f"  {i + 1}/{len(cases)}，剩余约 {el / (i + 1) * (len(cases) - i - 1) / 60:.1f} 分钟", end="\r")
    print()
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--ckpt_dir", default=None, help="评测目录下所有检查点（实验 1）")
    ap.add_argument("--baselines_only", action="store_true")
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--splits", nargs="+", default=["id", "ood"])
    ap.add_argument("--styles", nargs="+", default=["structured", "narrative"])
    ap.add_argument("--ks", nargs="+", type=int, default=[0, 1, 2, 4, 8, 12])
    ap.add_argument("--n_worlds", type=int, default=60, help="每个评测集用多少个世界")
    ap.add_argument("--targets", type=int, default=2, help="每个世界评几个目标年")
    ap.add_argument("--out", default="results")
    ap.add_argument("--device", default=None)
    args = ap.parse_args()
    os.environ["CIVLM_DATA"] = args.data_dir
    os.makedirs(args.out, exist_ok=True)

    ew = json.load(open(os.path.join(args.data_dir, "eval_worlds.json")))
    cases = []
    for sp in args.splits:
        cs = build_cases(ew[sp][:args.n_worlds], ew["years"], args.ks, args.styles, args.targets)
        for c in cs:
            c["split"] = sp
        cases += cs
    print(f"评测样本 {len(cases)} 个（{len(args.splits)} 个评测集 × {args.n_worlds} 世界 × {args.targets} 目标年 × "
          f"{len(args.ks)} 种 k × {len(args.styles)} 种文体）")

    if args.baselines_only:
        rows = summarize(cases, False)
        print_table(rows, False)
        plot(rows, os.path.join(args.out, "baselines.png"), "baselines", False)
        json.dump(rows, open(os.path.join(args.out, "baselines.json"), "w"), indent=1)
        return

    if args.ckpt_dir:
        ckpts = sorted(glob.glob(os.path.join(args.ckpt_dir, "*.pt")))
        curve = []
        for ck in ckpts:
            print(f"== {ck}")
            res = evaluate_ckpt(ck, cases, args)
            rows = summarize(res, True)
            step = int(re.findall(r"\d+", os.path.basename(ck))[-1])
            for r in rows:
                r["step"] = step
            curve += rows
        json.dump(curve, open(os.path.join(args.out, "learning_curve.json"), "w"), indent=1)
        plot_curve(curve, os.path.join(args.out, "learning_curve.png"))
        return

    res = evaluate_ckpt(args.ckpt, cases, args)
    rows = summarize(res, True)
    print_table(rows, True)
    name = os.path.splitext(os.path.basename(args.ckpt))[0]
    tag = os.path.basename(os.path.dirname(args.ckpt)) + "_" + name
    json.dump(rows, open(os.path.join(args.out, f"{tag}.json"), "w"), indent=1)
    with open(os.path.join(args.out, f"{tag}_samples.jsonl"), "w", encoding="utf-8") as f:
        for c in res:
            f.write(json.dumps({k: v for k, v in c.items() if k not in ("hist", "weather_cands", "header")},
                               ensure_ascii=False) + "\n")
    plot(rows, os.path.join(args.out, f"{tag}.png"), tag, True)


def plot_curve(curve, path):
    """实验 1：训练进程中，格式能力（parse）与世界规律能力（收成误差、粮食方向）分别何时出现。"""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for sp in sorted({r["split"] for r in curve}):
        for k in sorted({r["k"] for r in curve}):
            if k not in (0, 4, 12):
                continue
            rs = sorted([r for r in curve if r["split"] == sp and r["k"] == k and r["style"] == "structured"],
                        key=lambda r: r["step"])
            if not rs:
                continue
            st = [r["step"] for r in rs]
            axes[0].plot(st, [r["lm_parse"] for r in rs], label=f"{sp} k={k}")
            axes[1].plot(st, [r["lm_harvest_err"] for r in rs], label=f"{sp} k={k}")
            axes[2].plot(st, [r["lm_food_acc"] for r in rs], label=f"{sp} k={k}")
    for ax, t in zip(axes, ["format: parse rate", "harvest |log error|", "food direction accuracy"]):
        ax.set_title(t)
        ax.set_xlabel("training step")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    print(f"图 → {path}")


if __name__ == "__main__":
    main()


In [ ]:
%cd /content/civlm
from google.colab import files
!ls


## 2.（建议）把输出存到 Google Drive
Colab 断线后本地磁盘会清空。运行这一格后，检查点直接写进 Drive 的 `civlm_out/`，断线后也能用 `--resume` 继续训练。
不需要的话可以跳过。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/civlm_out', exist_ok=True)
if not os.path.islink('out'):
    !rm -rf out
    os.symlink('/content/drive/MyDrive/civlm_out', 'out')
!ls -la out

## 3. 生成数据
默认 3 万个文明，约 1 亿 token。核数少时会慢一些，屏幕上会显示用时。

In [ ]:
!python gen_data.py --n_worlds 30000
!cat data/meta.json

## 4. 参照线（不需要模型，1～2 分钟）

In [ ]:
!python eval_icl.py --baselines_only

## 5. 训练
A100 会自动使用 bf16。

- **26M**（先跑这个）：约 3 亿 token，A100 预计几十分钟，以屏幕显示为准。
- **100M**：等 26M 结果有意义后再跑，把下一格的 `SIZE` 改成 `"100m"`。

每 250 步会打印“预测 → / 真实 →”的对照。

In [ ]:
SIZE = "26m"   # 或 "100m"
if SIZE == "26m":
    !python pretrain.py --name civlm26m --batch_size 32 --max_steps 5000
else:
    !python pretrain.py --name civlm100m --n_layer 12 --n_head 12 --n_embd 768 --batch_size 32 --max_steps 8000 --lr 4e-4

断线后继续训练（需要第 2 步存到了 Drive）：重新运行第 1、2 步，第 3 步数据要重新生成（同样的参数会得到完全相同的数据），然后：
```
!python pretrain.py --name civlm26m --batch_size 32 --max_steps 5000 --resume
```

## 6. 打包下载
只打包本地评测需要的东西：模型检查点、分词器、评测世界清单和训练日志。不包含 train.bin。

In [ ]:
NAME = "civlm26m"   # 或 "civlm100m"
!rm -f /content/{NAME}_result.zip
!zip -r /content/{NAME}_result.zip out/{NAME} data/tokenizer.json data/eval_worlds.json data/meta.json results -x "*.bin"
!ls -lh /content/{NAME}_result.zip
files.download(f'/content/{NAME}_result.zip')

## （可选）在 Colab 上直接评测
A100 上评测更快。结果也会在 `results/` 里，重新运行第 6 步就能一起下载。

In [ ]:
!python eval_icl.py --ckpt out/{NAME}/best.pt
!python eval_icl.py --ckpt_dir out/{NAME}/ckpts --ks 0 4 12 --styles structured --n_worlds 30